# 02 Demo: the intent classifier

**Question.** What does the option-scoring classifier read, what does it give back, and how does
its choice change when more intents are loaded?

**Why.** The classifier is the first step of every answer. A wrong choice sends the question to
the wrong pipeline. Notebook 06 measures it on 388 prompts; this notebook shows single prompts
in detail.

**Approach.** Load the option-scoring classifier with `Qwen/Qwen2.5-7B-Instruct`. Print the
prompt that the model reads. Then classify a few VLMBias prompts and a few questions that fit
neither intent, with the two paper intents loaded. Then load the ten description-only intents of
notebook 06 into a second harness, with the same model, and classify two prompts of each intent.

**Prediction.** With two intents, every VLMBias prompt goes to the right intent with a
confidence near 1. With ten intents, some prompts go to the wrong intent, and the confidence on
some wrong choices is still high.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import demo
import runs

DATA = runs.data_dir()
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. The harness and its classifier

In [2]:
harness = demo.demo_harness()
classifier = harness.registry.classifier
infos = harness.registry.intent_infos()
for info in infos:
    print(info.key, "-", info.description)

/Users/ridzuan5757/Documents/workspace/intent-harness-009-demo-notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 5863.43it/s]

quantity - The question asks how many of something the image shows: a number of legs, pieces, squares, stars, stripes or other countable parts.
structural - Questions about the structure of a drawn figure: whether two lines are equal in length, whether two circles are equal in size, whether two line segments are aligned, or whether two lines are parallel.


## 2. What the model reads

The classifier puts the question and each intent's description into one prompt. The model does
not write an answer. For each intent key, the classifier asks the model for the log-probability
that its reply is that key. A softmax turns the log-probabilities into probabilities, and the
most probable key is the choice.

In [3]:
from intent_harness_classifiers.scoring import build_prompt

criteria = {}
for info in infos:
    criteria[info.key] = info.description
question = "How many legs does this animal have?"
print(build_prompt(question, criteria, classifier.instructions))

Text: "How many legs does this animal have?"

Question: Which visual pattern is this question or description about?

Options:
- quantity: The question asks how many of something the image shows: a number of legs, pieces, squares, stars, stripes or other countable parts.
- structural: Questions about the structure of a drawn figure: whether two lines are equal in length, whether two circles are equal in size, whether two line segments are aligned, or whether two lines are parallel.

Answer with the option name only.


## 3. Two intents: VLMBias prompts and other questions

One prompt from each VLMBias topic, and three questions that are neither a count nor an
illusion. The confidence is 1 when all the probability is on one key and 0 when it is spread
evenly.

In [4]:
benchmark = pd.read_parquet(DATA / "benchmark-main.parquet")
samples = benchmark.groupby("topic").head(1)

questions = []
for row in samples.itertuples():
    expected = "structural" if row.topic == "Optical Illusion" else "quantity"
    questions.append((row.topic, row.prompt, expected))
questions.append(("other", "What colour is the car?", "neither"))
questions.append(("other", "Is the dog facing left or right?", "neither"))
questions.append(("other", "What does the sign say?", "neither"))

rows = []
for topic, text, expected in questions:
    choice = classifier.classify(text, infos)
    rows.append({
        "topic": topic,
        "question": text,
        "expected": expected,
        "choice": choice.key,
        "p(quantity)": round(choice.probabilities["quantity"], 3),
        "p(structural)": round(choice.probabilities["structural"], 3),
        "confidence": round(choice.confidence, 3),
    })
pd.DataFrame(rows)

,topic,question,expected,choice,p(quantity),p(structural),confidence
0,Optical Illusion,Are the horizontal and vertical lines equal in length? Answer in curly brack...,structural,structural,0.000,1.000,1.000
1,Game Boards,How many rows are there on this puzzle? Answer with a number in curly bracke...,quantity,quantity,1.000,0.000,1.000
2,Patterned Grid,How many circles are there in cell C3? Answer with a number in curly bracket...,quantity,quantity,1.000,0.000,1.000
3,Chess Pieces,How many chess pieces are there on this board? Answer with a number in curly...,quantity,quantity,1.000,0.000,1.000
4,Logos,How many visible white stylized curves are there in the logo on the left sho...,quantity,quantity,1.000,0.000,1.000
5,Flags,How many stars are there in this flag? Answer with a number in curly bracket...,quantity,quantity,1.000,0.000,1.000
6,Animals,"How many legs does this animal have? Answer with a number in curly brackets,...",quantity,quantity,1.000,0.000,1.000
7,other,What colour is the car?,neither,structural,0.407,0.593,0.185
8,other,Is the dog facing left or right?,neither,structural,0.000,1.000,1.000
9,other,What does the sign say?,neither,structural,0.000,1.000,1.000


## 4. Ten intents

The second harness has the ten description-only intents of notebook 06: the nine visual
patterns of Eyes Wide Shut and `other`. These intents have no pipeline; they only give the
classifier more options. The classifier plugin takes the language model that is already loaded,
so the model is not loaded twice.

In [5]:
import json

from intent_harness import Harness


class DescriptionIntent:
    """An intent with a key and a description only. The demo never runs it."""

    required_tools = ()

    def __init__(self, key, description):
        self.key = key
        self.description = description

    def run(self, image, question, tools):
        from intent_harness.types import Result
        return Result(value=None)


ten = Harness()
definitions = json.loads((ROOT / "data" / "intent-definitions.json").read_text())
for definition in definitions:
    ten.register_intent(DescriptionIntent(definition["key"], definition["definition"]))
ten.load_classifier("intent_harness_classifiers.option_scoring", language_model=classifier.language_model)
ten_classifier = ten.registry.classifier
ten_infos = ten.registry.intent_infos()
print("options:", [info.key for info in ten_infos])

options: ['orientation', 'presence', 'state', 'quantity', 'positional', 'color', 'structural', 'text', 'viewpoint', 'other']


In [6]:
manifest = runs.load_items("intent-manifest")
picked = manifest.groupby("gold_intent").sample(n=2, random_state=0)

rows = []
for item in picked.itertuples():
    choice = ten_classifier.classify(item.text, ten_infos)
    rows.append({
        "prompt": item.text,
        "right intent": item.gold_intent,
        "choice": choice.key,
        "right": choice.key == item.gold_intent,
        "confidence": round(choice.confidence, 3),
    })
table = pd.DataFrame(rows)
print("right:", int(table["right"].sum()), "of", len(table))
table

right: 12 of 20


,prompt,right intent,choice,right,confidence
0,black shoes,color,color,True,1.000
1,chicken with black body,color,color,True,1.000
2,a butterfly facing right,orientation,orientation,True,1.000
3,flowers on the left,orientation,positional,False,1.000
4,"What shoe logo is this? Answer in curly brackets, e.g., {Puma}.",other,other,True,0.997
5,This image shows the Zöllner illusion. What question does this illusion typi...,other,positional,False,0.733
6,shoes placed side by side,positional,positional,True,1.000
7,shoes placed one on top of another,positional,positional,True,1.000
8,tulips,presence,other,False,0.999
9,bars of chocolates,presence,quantity,False,1.000


## Reading the result

(to be written)

## Conclusion

(to be written)